# DiffSBDD — Molecule Generation Setup

This notebook documents the setup and execution of DiffSBDD for 
de novo molecule generation targeting the A2A adenosine receptor (PDB: 4EIY).

DiffSBDD uses a 3D equivariant diffusion model — unlike Pocket2Mol's 
atom-by-atom approach, DiffSBDD generates all atoms simultaneously by 
denoising from Gaussian noise.

---
> **Repository:** https://github.com/arneschneuing/DiffSBDD  
> **Paper:** Schneuing et al., arXiv:2210.13695 (2022)  
> **Checkpoint:** https://zenodo.org/record/8183747  
> **Output:** `outputs/diffsbdd/DiffSBDD_ligands.sdf`

## 1. Repository Setup

```bash
# Clone DiffSBDD repository into project folder
cd ~/fhnw/fhnw_projects/2026SS_ai_drug_discovery/
git clone https://github.com/arneschneuing/DiffSBDD.git DiffSBDD-main
```

> The repository is stored in `DiffSBDD-main/` — this name is used throughout 
> the project for consistency.

## 2. Environment Setup

DiffSBDD requires its own conda environment — separate from Pocket2Mol and 
`ai_dd_analysis` to avoid dependency conflicts.

```bash
# Create conda environment from DiffSBDD's environment file
conda env create -f DiffSBDD-main/environment.yaml

# Activate the environment
conda activate diffsbdd
```

> **Note:** Environment setup may take 10–15 minutes depending on internet speed.
> The environment includes PyTorch, torch-geometric, and other deep learning dependencies.

## 3. Pretrained Checkpoint

DiffSBDD requires a pretrained model checkpoint to generate molecules.
Download from Zenodo and place in the correct location:

```bash
# Download checkpoint (requires ~500MB disk space)
# Source: https://zenodo.org/record/8183747

# Place checkpoint here:
# DiffSBDD-main/checkpoints/crossdocked_fullatom_cond.ckpt
```

> The checkpoint was trained on the CrossDocked2020 dataset — 
> ~22M protein-ligand poses from ~5000 proteins.

## 4. macOS Adaptation

DiffSBDD environment has been adapted for macOS ARM64 (Apple Silicon) 
compatible dependency resolution. 

> See `DiffSBDD-main/environment.yaml` for the adapted environment file.
> Original CUDA dependencies were replaced with CPU-compatible alternatives.

## 5. Generate Molecules with DiffSBDD

DiffSBDD uses inpainting-based generation — it takes a reference ligand fragment
and generates new molecules that fit the same binding pocket.

```bash
conda run -n diffsbdd python DiffSBDD-main/inpaint.py \
  DiffSBDD-main/checkpoints/crossdocked_fullatom_cond.ckpt \
  --pdbfile data/4eiy_clean.pdb \
  --outfile outputs/diffsbdd/4eiy_inpaint.sdf \
  --ref_ligand DiffSBDD-main/example/5ndu_C_8V2.sdf \
  --fix_atoms DiffSBDD-main/example/fragments.sdf \
  --center pocket \
  --n_samples 100
```

> **Parameters:**
> - `--pdbfile` — cleaned receptor structure (ligand removed)
> - `--outfile` — output SDF file path
> - `--ref_ligand` — reference ligand for inpainting mode
> - `--fix_atoms` — fixed fragment atoms to keep during generation
> - `--center pocket` — center generation on the binding pocket
> - `--n_samples 100` — generate 100 molecules

## 6. Results

| Metric | Value |
|---|---|
| Molecules requested | 100 |
| Valid molecules generated | 116 |
| Generation runs | 3 (outputs combined) |
| Output file | `outputs/diffsbdd/DiffSBDD_ligands.sdf` |

> **Note:** Three separate runs were performed and combined into a single SDF file
> using `combine_sdfs.py`. Full molecular property analysis in 
> `03b_diffsbdd_molecule_analysis.ipynb`.